In [6]:
# Roll A
# Data Loading

# import necessary libraries
from dotenv import load_dotenv
import os
from supabase import create_client
import pandas as pd

# Load environment variables and create Supabase client
load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Function to fetch data from Supabase in chunks
def get_data(table_name):
    data = []
    page_size = 1000
    page = 0
    while True:
        response = supabase.table(table_name).select("*").range(page * page_size, (page + 1) * page_size - 1).execute()
        data.extend(response.data)
        if len(response.data) < page_size:
            break
        page += 1
    return pd.DataFrame(data)

# Fetch data from Supabase tables
df_sales = get_data("sales")
df_customers = get_data("customers")

# Merge the sales and customers dataframes on customer_id
df=pd.merge(df_sales, df_customers[['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city']], on='customer_id', how='left')

# print shape and head of sales dataframe
print("--sales shape--:", df_sales.shape)
print("--sales head--:", df_sales.head())

# print shape and head of customers dataframe
print("--customers shape--:", df_customers.shape)
print("--customers head--:", df_customers.head())

# print shape, head, and data types of merged dataframe
print("--merged shape--:", df.shape)
print("--merged head--:", df.head())
print("--merged data types--:", df.dtypes)

# Roll B
# Data Cleaning

# Data already cleaned
# Testing the data
print("Duplikaadid:", df.duplicated(subset=['invoice_id']).sum())
# check for missing values
print("NULL-id:\n", df.isnull().sum())
# drop rows with missing values in customer_id, sale_date, or total_price
df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])
# convert sale_date to datetime
df['sale_date'] = pd.to_datetime(df['sale_date'])
# remove rows with sale_date after year 2024
df = df[df['sale_date'].dt.year < 2025]
# check for negative or zero total_price values
print("Negatiivseid või null total_price väärtusi:",
      (df['total_price'] <= 0).sum())
# drop rows with negative or zero total_price values
df = df[df['total_price'] > 0]
# cleaning report
print("\nPUHASTUSRAPORT")
print("----------------------")
print("Lõplik shape:", df.shape)
print("Unikaalseid kliente:", df['customer_id'].nunique())
print("Kuupäevavahemik:",
      df['sale_date'].min(),
      "kuni",
      df['sale_date'].max())


--sales shape--: (10118, 12)
--sales head--:    id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00       2588.0   
1   2        2  INV-202301-00002  2023-01-16T00:00:00       4338.0   
2   3        3  INV-202301-00003  2023-01-05T00:00:00       4673.0   
3   4        4  INV-202301-00004  2023-01-02T00:00:00       4677.0   
4   5        5  INV-202301-00005  2023-01-13T00:00:00       2390.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1274         2      234.79       469.58    pood        Tallinn   
1        1207         2      241.13       482.26    pood          Pärnu   
2        1264         1      258.46       221.19    pood          Pärnu   
3        1341         3       45.21       135.63    pood          Tartu   
4        1284         1       99.57        99.57    pood          Tartu   

  payment_method  
0          kaart  
1      järelmaks  
2      järelmaks  
3      